# 02_run-multi-match-prediction

This notebook simulates all selected team-vs-team matchups and builds matrix-style outputs for:

- row-team win probability
- draw probability
- column-team win probability
- most common exact score
- recommended score tip
- expected points for the recommended tip


In [ ]:
import os

phase = 'round_of_32'
os.environ['WORLD_CUP_PHASE'] = phase

print(f"Set WORLD_CUP_PHASE to '{phase}'")

In [ ]:
import itertools
import os
import random
import time
import warnings
from contextlib import contextmanager
from pathlib import Path

import joblib
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.notebook import tqdm

from utils.simulation import (
    build_matchup_detail_tables,
    find_optimal_tip_from_simulations,
    mirror_matchup_detail_tables,
    simulate_match_many,
)

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)

In [ ]:
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

@contextmanager
def tqdm_joblib(tqdm_object):
    """Patch joblib to report finished batches into a tqdm progress bar."""
    original_callback = joblib.parallel.BatchCompletionCallBack

    class TqdmBatchCompletionCallback(original_callback):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(self.batch_size)
            return super().__call__(*args, **kwargs)

    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = original_callback
        tqdm_object.close()


def format_elapsed_seconds(total_seconds):
    total_seconds = max(0.0, float(total_seconds))
    minutes, seconds = divmod(total_seconds, 60)
    hours, minutes = divmod(int(minutes), 60)
    if hours:
        return f'{hours:d}h {minutes:d}m {seconds:0.1f}s'
    if minutes:
        return f'{minutes:d}m {seconds:0.1f}s'
    return f'{seconds:0.1f}s'

## Configuration

Adjust `phase` and `n_simulations_per_match` depending on the use case.

Scoreline sampling now uses Poisson only.

For a first run, keep the team list small and use fewer simulations. For final outputs, increase the number of simulations and include all relevant teams.

In [ ]:
n_simulations_per_match = 10_000 # Number of simulations to run per matchup
max_tip_goals = 6 # Cutoff for maximum goals to consider in the tip optimization
use_cached_results = False # Use pre-existing results if available, otherwise run simulations
n_jobs = max(1, (os.cpu_count() or 1) - 1) # Set to 1 for sequential execution
parallel_backend = 'loky' # 'loky' backend uses separate worker processes and works well for CPU-bound tasks

scoreline_method_suffix = ''
dir_out = Path(f'simulation_results/{phase}')
dir_out.mkdir(parents=True, exist_ok=True)
path_cache = dir_out / f'prediction_matrix_long_{phase}{scoreline_method_suffix}.csv'

detail_cache_paths = {
    'score_distribution': (
        dir_out / f'matchup_score_distribution_{phase}{scoreline_method_suffix}.csv'
    ),
    'outcome_distribution': (
        dir_out / f'matchup_outcome_distribution_{phase}{scoreline_method_suffix}.csv'
    ),
    'goal_diff_distribution': (
        dir_out / f'matchup_goal_diff_distribution_{phase}{scoreline_method_suffix}.csv'
    ),
    'tip_rank': dir_out / f'matchup_tip_rank_{phase}{scoreline_method_suffix}.csv',
}
detail_table_names = list(detail_cache_paths)


In [ ]:
from utils.simulation import groups
from utils.runtime_data import TEAMS_ROUND_OF_32, TEAMS_ROUND_OF_16 #, TEAMS_QUARTERFINAL, TEAMS_SEMIFINAL, TEAMS_THIRD_PLACE, TEAMS_FINAL

# Select teams per phase
dir_teams = {
    'group': sorted(set(team for group_teams in groups.values() for team in group_teams)),
    'round_of_32': sorted(TEAMS_ROUND_OF_32),
    'round_of_16': sorted(TEAMS_ROUND_OF_16),
    'quarterfinal': None, # tbd
    'semifinal': None, # tbd
    'third_place': None, # tbd
    'final': None, # tbd
}
teams = sorted(dir_teams[phase])
n_ordered_matrix_cells = len(teams) * (len(teams) - 1)
n_simulated_matchups = len(teams) * (len(teams) - 1) // 2

print(f'Teams: {len(teams)}')
print(f'Simulated matchups: {n_simulated_matchups:,}')
print(f'Simulations per matchup: {n_simulations_per_match:,}')
print(f'Parallel workers: {n_jobs}')
print(
    'Total simulated matches: '
    f'{n_simulated_matchups * n_simulations_per_match:,}'
)

## Matchup summary helper

This helper runs one row-team vs column-team simulation and returns both the matrix-ready summary row and the detailed distributions needed to rebuild the single-match 2x2 view later.

In [ ]:
def simulate_matchup_tables(
    team_a,
    team_b,
    phase='group',
    n_simulations=3000,
    max_tip_goals=6,
    use_tqdm=False
):
    """Simulate one matchup and return summary and detail tables.

    Args:
        team_a: Row team.
        team_b: Column team.
        phase: Prediction-game phase
        n_simulations: Number of Monte Carlo simulations.
        max_tip_goals: Highest goals to consider for candidate score tips.

    Returns:
        Dictionary with summary and detail tables for this matchup.
    """
    df_simulations = simulate_match_many(
        team_a=team_a,
        team_b=team_b,
        phase=phase,
        n_simulations=n_simulations,
        use_tqdm=use_tqdm
    )

    df_tiprank = find_optimal_tip_from_simulations(
        df_simulations=df_simulations,
        team_a=team_a,
        team_b=team_b,
        phase=phase,
        max_tip_goals=max_tip_goals,
    )

    return build_matchup_detail_tables(
        df_simulations=df_simulations,
        df_tiprank=df_tiprank,
        team_a=team_a,
        team_b=team_b,
    )


## Run all matchups

The matrix is row-oriented: `team_a` is the row team and `team_b` is the column opponent.

Each unordered matchup is simulated once, then mirrored into the opposite direction.
For example, `Switzerland vs Brazil` is simulated directly, and `Brazil vs Switzerland`
is derived by swapping teams, probabilities, goals, and score tips.

This avoids running the same neutral matchup twice and keeps the matrix symmetric.

In [ ]:
detail_table_display_names = {
    'score_distribution': 'score distributions',
    'outcome_distribution': 'outcome distributions',
    'goal_diff_distribution': 'goal-difference distributions',
    'tip_rank': 'tip ranks',
}

In [ ]:
detail_cache_ready = all(path.exists() for path in detail_cache_paths.values())
df_matrix_long = None
df_matchup_details = None

def run_matchup_pair(team_a, team_b):
    matchup_tables = simulate_matchup_tables(
        team_a=team_a,
        team_b=team_b,
        phase=phase,
        n_simulations=n_simulations_per_match,
        max_tip_goals=max_tip_goals,
    )
    mirrored_matchup_tables = mirror_matchup_detail_tables(matchup_tables)
    return matchup_tables, mirrored_matchup_tables

matrix_run_started_at = time.perf_counter()
matrix_result_source = 'cache'

if use_cached_results and path_cache.exists() and detail_cache_ready:
    df_matrix_long = pd.read_csv(path_cache)
    df_matchup_details = {
        name: pd.read_csv(path)
        for name, path in detail_cache_paths.items()
    }
    print(f'Loaded cached results from: {path_cache}')
    for name, path in detail_cache_paths.items():
        print(f'Loaded cached {detail_table_display_names[name]} from: {path}')
else:
    matrix_result_source = 'simulation'
    if use_cached_results and path_cache.exists() and not detail_cache_ready:
        print('Summary cache exists, but matchup detail caches are incomplete. Re-running simulations.')

    rows = []
    detail_rows = {name: [] for name in detail_table_names}
    matchups = list(itertools.combinations(teams, 2))

    if n_jobs == 1:
        print('Running simulations sequentially.')
        matchup_results = [
            run_matchup_pair(team_a, team_b)
            for team_a, team_b in tqdm(matchups, desc='Simulating matchups', unit='matchup')
        ]
    else:
        print(f'Running simulations in parallel with {n_jobs} workers ({parallel_backend} backend).')
        progress_bar = tqdm(total=len(matchups), desc='Simulating matchups', unit='matchup')
        with tqdm_joblib(progress_bar):
            matchup_results = Parallel(n_jobs=n_jobs, backend=parallel_backend)(
                delayed(run_matchup_pair)(team_a, team_b)
                for team_a, team_b in matchups
            )

    for matchup_tables, mirrored_matchup_tables in matchup_results:
        rows.append(matchup_tables['summary'])
        rows.append(mirrored_matchup_tables['summary'])

        for name in detail_table_names:
            detail_rows[name].append(matchup_tables[name])
            detail_rows[name].append(mirrored_matchup_tables[name])

    df_matrix_long = pd.DataFrame(rows)
    df_matchup_details = {
        name: pd.concat(tables, ignore_index=True)
        for name, tables in detail_rows.items()
    }

    df_matrix_long.to_csv(path_cache, index=False)
    print(f'Saved long-format results to: {path_cache}')

    for name, path in detail_cache_paths.items():
        df_matchup_details[name].to_csv(path, index=False)
        print(f'Saved {detail_table_display_names[name]} to: {path}')

matrix_run_elapsed_seconds = time.perf_counter() - matrix_run_started_at
n_matchups_processed = len(df_matrix_long) // 2
avg_seconds_per_matchup = (
    matrix_run_elapsed_seconds / n_matchups_processed
    if n_matchups_processed
    else float('nan')
)

print(
    f"Matrix data ready from {matrix_result_source} in "
    f"{format_elapsed_seconds(matrix_run_elapsed_seconds)} "
    f"({avg_seconds_per_matchup:0.2f}s per simulated matchup)."
)

df_matrix_long.head()

## Create matrix tables

In [ ]:
def make_matrix(value_column):
    """Return a team-vs-team matrix for a selected value column."""
    return (
        df_matrix_long
        .pivot(index='team_a', columns='team_b', values=value_column)
        .reindex(index=teams, columns=teams)
    )


df_row_win_matrix = make_matrix('team_a_win_probability')
df_draw_matrix = make_matrix('draw_probability')
df_col_win_matrix = make_matrix('team_b_win_probability')
df_tip_matrix = make_matrix('recommended_tip')
df_tip_points_matrix = make_matrix('recommended_tip_expected_points')
df_score_matrix = make_matrix('most_common_score')
df_score_probability_matrix = make_matrix('most_common_score_probability')
df_avg_goal_diff_matrix = make_matrix('avg_goal_difference')

df_row_win_matrix.head()


## Save matrix outputs

This writes the matrix CSVs used for high-level comparison. Matchup-level detail distributions are already persisted in the previous step for the interactive notebook.

In [ ]:
matrix_outputs = {
    f'row_team_win_probability_matrix_{phase}{scoreline_method_suffix}.csv': df_row_win_matrix,
    f'draw_probability_matrix_{phase}{scoreline_method_suffix}.csv': df_draw_matrix,
    f'column_team_win_probability_matrix_{phase}{scoreline_method_suffix}.csv': df_col_win_matrix,
    f'recommended_tip_matrix_{phase}{scoreline_method_suffix}.csv': df_tip_matrix,
    f'recommended_tip_expected_points_matrix_{phase}{scoreline_method_suffix}.csv': df_tip_points_matrix,
    f'most_common_score_matrix_{phase}{scoreline_method_suffix}.csv': df_score_matrix,
    f'most_common_score_probability_matrix_{phase}{scoreline_method_suffix}.csv': (
        df_score_probability_matrix
    ),
    f'avg_goal_difference_matrix_{phase}{scoreline_method_suffix}.csv': df_avg_goal_diff_matrix,
}

for filename, df_output in matrix_outputs.items():
    path_out = dir_out / filename
    df_output.to_csv(path_out)
    print(f'Saved: {path_out}')


## Plot: win probability matrix

Cell values show the probability that the row team beats the column team.


In [ ]:
n_teams = len(df_tip_matrix)
cell_size = 0.25
margin = 4
fig_size = n_teams * cell_size + margin

In [ ]:
plt.figure(figsize=(fig_size, fig_size))

sns.heatmap(
    df_row_win_matrix,
    annot=True,
    fmt='.0f',
    cmap='RdYlGn',
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Win probability (%)'},
)

plt.title(
    f'Prediction Matrix - Win Probability'
    f'\n(phase={phase}, n_sim={n_simulations_per_match:,})',
    fontsize=16,
    # fontweight='bold',
)
plt.xlabel('Opponent', fontsize=16)
plt.ylabel('Team to tip', fontsize=16)
plt.tight_layout()
plt.show()


## Plot: recommended tip matrix

Cell text shows the recommended score tip. Colour shows the expected points of that tip.


In [ ]:
fig, ax = plt.subplots(figsize=(fig_size, fig_size))

sns.heatmap(
    df_tip_points_matrix,
    annot=df_tip_matrix,
    fmt='',
    cmap='RdYlGn',
    linewidths=0.25,
    linecolor='white',
    square=True,
    ax=ax,
    cbar_kws={
        'label': 'Expected points of recommended tip',
        'shrink': 0.75,
        'pad': 0.02,
    },
    annot_kws={
        'fontsize': 8,
        'ha': 'center',
        'va': 'center',
    },
)

ax.set_title(
    f'Prediction Matrix - Recommended Tip'
    f'\n(phase={phase}, n_sim={n_simulations_per_match:,})',
    fontsize=16,
    # fontweight='bold',
    pad=18,
)

ax.set_xlabel('Opponent', fontsize=12)
ax.set_ylabel('Team to tip', fontsize=12)

ax.set_xticklabels(
    ax.get_xticklabels(),
    rotation=45,
    ha='right',
    fontsize=10,
)

ax.set_yticklabels(
    ax.get_yticklabels(),
    rotation=0,
    fontsize=10,
)

plt.tight_layout()
plt.show()

## Plot: most common exact score matrix

Cell text shows the most common exact score. Colour shows how often that exact score appears in the simulations.


In [ ]:
fig, ax = plt.subplots(figsize=(fig_size, fig_size))

sns.heatmap(
    df_score_probability_matrix,
    annot=df_score_matrix,
    fmt='',
    cmap='RdYlGn',
    linewidths=0.25,
    linecolor='white',
    square=True,
    ax=ax,
    cbar_kws={
        'label': 'Probability of most common score (%)',
        'shrink': 0.75,
        'pad': 0.02,
    },
    annot_kws={
        'fontsize': 8,
        'ha': 'center',
        'va': 'center',
    },
)

ax.set_title(
    f'Prediction Matrix - Predicted Score'
    f'\n(phase={phase}, n_sim={n_simulations_per_match:,})',
    fontsize=16,
    # fontweight='bold',
    pad=18,
)

ax.set_xlabel('Opponent', fontsize=12)
ax.set_ylabel('Team to tip', fontsize=12)

ax.set_xticklabels(
    ax.get_xticklabels(),
    rotation=45,
    ha='right',
    fontsize=10,
)

ax.set_yticklabels(
    ax.get_yticklabels(),
    rotation=0,
    fontsize=10,
)

plt.tight_layout()
plt.show()

## Plot: average goal-difference matrix

Positive values favour the row team. Negative values favour the column team.


In [ ]:
goal_diff_abs_max = np.nanmax(np.abs(df_avg_goal_diff_matrix.values))

fig, ax = plt.subplots(figsize=(fig_size, fig_size))

sns.heatmap(
    df_avg_goal_diff_matrix,
    annot=True,
    fmt='.1f',
    vmin=-goal_diff_abs_max,
    vmax=goal_diff_abs_max,
    cmap='RdYlGn',
    linewidths=0.25,
    linecolor='white',
    square=True,
    ax=ax,
    cbar_kws={
        'label': 'Average goal difference',
        'shrink': 0.75,
        'pad': 0.02,
    },
    annot_kws={
        'fontsize': 8,
        'ha': 'center',
        'va': 'center',
    },
)

ax.set_title(
    f'Prediction Matrix - Expected Goal Difference'
    f'\n(phase={phase}, n_sim={n_simulations_per_match:,})',
    fontsize=16,
    # fontweight='bold',
    pad=18,
)

ax.set_xlabel('Opponent', fontsize=12)
ax.set_ylabel('Team to tip', fontsize=12)

ax.set_xticklabels(
    ax.get_xticklabels(),
    rotation=45,
    ha='right',
    fontsize=10,
)

ax.set_yticklabels(
    ax.get_yticklabels(),
    rotation=0,
    fontsize=10,
)

plt.tight_layout()
plt.show()

## Team ranking summary

This summarizes each row team's average performance across all selected opponents.


In [ ]:
df_team_strength = (
    df_matrix_long
    .groupby('team_a')
    .agg(
        avg_win_probability=('team_a_win_probability', 'mean'),
        avg_draw_probability=('draw_probability', 'mean'),
        avg_loss_probability=('team_b_win_probability', 'mean'),
        avg_recommended_tip_expected_points=(
            'recommended_tip_expected_points',
            'mean',
        ),
        avg_goals_for=('avg_goals_a', 'mean'),
        avg_goals_against=('avg_goals_b', 'mean'),
        avg_goal_difference=('avg_goal_difference', 'mean'),
    )
    .sort_values('avg_win_probability', ascending=False)
)

df_team_strength


In [ ]:
df_team_strength_plot = df_team_strength.reset_index()

plt.figure(figsize=(10, max(6, len(df_team_strength_plot) * 0.35)))

sns.barplot(
    data=df_team_strength_plot,
    x='avg_win_probability',
    y='team_a',
    color='#1f77b4',
)

plt.title(
    f'Avg. Win Probability Across Selected Matchups '
    f'\n(phase={phase}, n_sim={n_simulations_per_match:,})',
    fontsize=15,
    # fontweight='bold',
)
plt.xlabel('Average win probability (%)')
plt.ylabel('Team')
plt.grid(axis='x', color='#e6e6e6')
plt.tight_layout()
plt.show()


## Inspect one matchup

In [ ]:
inspect_team_a = 'Switzerland'
inspect_team_b = 'Brazil'

df_matrix_long[
    (df_matrix_long['team_a'] == inspect_team_a)
    & (df_matrix_long['team_b'] == inspect_team_b)
]


## Final summary

In [ ]:
print('Prediction matrix complete.')
print(f'Teams: {len(teams)}')
print('Scoreline method: poisson')
print(f'Ordered matchups: {len(df_matrix_long):,}')
print(f'Simulations per matchup: {n_simulations_per_match:,}')
print(f'Total simulated matches: {len(df_matrix_long) * n_simulations_per_match:,}')
print()
print('Top teams by average win probability:')
display(df_team_strength.head(10))
